In [27]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

<h2>Preparing the data</h2>

In [3]:
import kagglehub

In [4]:
path = kagglehub.dataset_download("fanbyprinciple/iot-device-identification")

In [5]:
data_train = pd.read_csv(path + "/iot_device_train.csv")
data_test = pd.read_csv(path + "/iot_device_test.csv")

In [6]:
data_all = pd.concat([data_train, data_test], axis=0, ignore_index=True)

In [10]:
data_all = data_all.sample(frac=1, random_state=42).reset_index(drop=True)

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
le = LabelEncoder()
data_all['device_category_label'] = le.fit_transform(data_all['device_category'])

In [39]:
data_all['device_category_label']

0       3
1       0
2       8
3       8
4       4
       ..
1895    2
1896    3
1897    2
1898    5
1899    2
Name: device_category_label, Length: 1900, dtype: int32

In [15]:
data_all.columns

Index(['ack', 'ack_A', 'ack_B', 'bytes', 'bytes_A', 'bytes_A_B_ratio',
       'bytes_B', 'ds_field_A', 'ds_field_B', 'duration',
       ...
       'suffix_is_com', 'suffix_is_com.sg', 'suffix_is_else',
       'suffix_is_empty_char_value', 'suffix_is_googleapis.com',
       'suffix_is_net', 'suffix_is_org', 'suffix_is_unresolved',
       'device_category', 'device_category_label'],
      dtype='object', length=299)

In [19]:
corr = data_all.drop(axis=0, columns='device_category').corr().abs()['device_category_label'].sort_values(ascending=False)

In [20]:
print(corr)

device_category_label       1.000000
reset                       0.404447
reset_A                     0.396896
ttl_B_thirdQ                0.345751
ttl_B_median                0.338601
                              ...   
suffix_is_co.il                  NaN
suffix_is_com.sg                 NaN
suffix_is_googleapis.com         NaN
suffix_is_net                    NaN
suffix_is_org                    NaN
Name: device_category_label, Length: 298, dtype: float64


In [43]:
dummies = pd.get_dummies(data_all['device_category'], dtype=int)
print(dummies)

      TV  baby_monitor  lights  motion_sensor  security_camera  \
0      0             0       0              1                0   
1      1             0       0              0                0   
2      0             0       0              0                0   
3      0             0       0              0                0   
4      0             0       0              0                1   
...   ..           ...     ...            ...              ...   
1895   0             0       1              0                0   
1896   0             0       0              1                0   
1897   0             0       1              0                0   
1898   0             0       0              0                0   
1899   0             0       1              0                0   

      smoke_detector  socket  thermostat  watch  water_sensor  
0                  0       0           0      0             0  
1                  0       0           0      0             0  
2              

In [44]:
categories = dummies.columns.tolist()
print(categories)

['TV', 'baby_monitor', 'lights', 'motion_sensor', 'security_camera', 'smoke_detector', 'socket', 'thermostat', 'watch', 'water_sensor']


In [24]:
data_X = data_all.drop(columns=['device_category', 'device_category_label'])
print(data_X)

        ack  ack_A  ack_B    bytes  bytes_A  bytes_A_B_ratio  bytes_B  \
0        48     18     30     1382      654         0.898350      728   
1         9      5      5     1213      743         1.142203      668   
2         9      5      5     1213      743         1.806874      668   
3         9      5      5     1213      743         1.806874      668   
4        12      6      6     1548      645         0.714285      903   
...     ...    ...    ...      ...      ...              ...      ...   
1895      0      0      0      240        0         0.000000      240   
1896  24718  12359  12359  1483080   741540         1.000000   741540   
1897      0      0      0      240        0         0.000000      240   
1898     14      7      7     2411     1317         1.203838     1094   
1899      0      0      0      240        0         0.000000      240   

      ds_field_A  ds_field_B    duration  ...  suffix_is_cloudfront.net  \
0              0           0      1.7060  ...   

In [45]:
corr_per_category = []
for category in categories:
    corr_per_category.append(pd.concat([data_X, dummies[category]], axis=1).corr().abs()[category].sort_values(ascending=False))

In [48]:
corr_per_category_top = [corr_per_category[i].head(10) for i in range(len(corr_per_category))]
print(corr_per_category_top)

[TV              1.000000
ttl_B_min       0.496075
reset_B         0.370424
ttl_A_avg       0.308538
ttl_A_firstQ    0.308538
ttl_A_max       0.308538
ttl_A_median    0.308538
ttl_A_min       0.308538
ttl_A_thirdQ    0.308538
ttl_B_median    0.287217
Name: TV, dtype: float64, baby_monitor             1.000000
reset_A                  0.660903
reset                    0.565487
ttl_B_stdev              0.517958
ttl_B_max                0.482812
ttl_B_thirdQ             0.469859
ttl_B_avg                0.434098
ttl_B_firstQ             0.424674
ttl_B_median             0.421507
packet_size_B_entropy    0.404990
Name: baby_monitor, dtype: float64, lights                         1.000000
packet_inter_arrivel_min       0.581965
B_port_is_8080                 0.577076
packet_inter_arrivel_firstQ    0.575378
ttl_min                        0.574552
ttl_firstQ                     0.573342
ds_field_B                     0.572415
packet_inter_arrivel_A_min     0.561040
packet_inter_arrivel_median

In [81]:
# turn corr_per_category into a dictionary
corr_per_category_top_dict = {}
for i in range(len(categories)):
    corr_per_category_top_dict[categories[i]] = corr_per_category_top[i]
print(corr_per_category_top_dict)

{'TV': TV              1.000000
ttl_B_min       0.496075
reset_B         0.370424
ttl_A_avg       0.308538
ttl_A_firstQ    0.308538
ttl_A_max       0.308538
ttl_A_median    0.308538
ttl_A_min       0.308538
ttl_A_thirdQ    0.308538
ttl_B_median    0.287217
Name: TV, dtype: float64, 'baby_monitor': baby_monitor             1.000000
reset_A                  0.660903
reset                    0.565487
ttl_B_stdev              0.517958
ttl_B_max                0.482812
ttl_B_thirdQ             0.469859
ttl_B_avg                0.434098
ttl_B_firstQ             0.424674
ttl_B_median             0.421507
packet_size_B_entropy    0.404990
Name: baby_monitor, dtype: float64, 'lights': lights                         1.000000
packet_inter_arrivel_min       0.581965
B_port_is_8080                 0.577076
packet_inter_arrivel_firstQ    0.575378
ttl_min                        0.574552
ttl_firstQ                     0.573342
ds_field_B                     0.572415
packet_inter_arrivel_A_min     0.56

In [86]:
# Get all unique feature names across all categories
features = set()
for i in range(len(categories)):
    top_corr = corr_per_category_top[i]
    if categories[i] in top_corr.index:
        top_corr = top_corr.drop(categories[i])
    features.update(top_corr.index)
features = list(features)
# order elements of all_features alphabetically
features.sort()
print(features)
print(len(features))

['B_port_is_11095', 'B_port_is_8080', 'domain_is_else', 'ds_field_B', 'http_count_host', 'http_count_req_content_type', 'http_count_resp_content_type', 'http_count_user_agents', 'http_has_req_content_type', 'http_has_resp_content_type', 'http_has_user_agent', 'is_http', 'packet_inter_arrivel_A_min', 'packet_inter_arrivel_B_min', 'packet_inter_arrivel_firstQ', 'packet_inter_arrivel_median', 'packet_inter_arrivel_min', 'packet_size_A_avg', 'packet_size_A_stdev', 'packet_size_A_thirdQ', 'packet_size_A_var', 'packet_size_B_avg', 'packet_size_B_entropy', 'packet_size_B_thirdQ', 'packets_A_B_ratio', 'reset', 'reset_A', 'reset_B', 'ssl_count_client_ciphersuites', 'ssl_dom_server_name_alexaRank', 'ssl_ratio_client_elliptic_curves', 'ssl_req_bytes_stdev', 'ttl_A_avg', 'ttl_A_firstQ', 'ttl_A_max', 'ttl_A_median', 'ttl_A_min', 'ttl_A_thirdQ', 'ttl_B_avg', 'ttl_B_firstQ', 'ttl_B_max', 'ttl_B_median', 'ttl_B_min', 'ttl_B_stdev', 'ttl_B_thirdQ', 'ttl_entropy', 'ttl_firstQ', 'ttl_max', 'ttl_median', 

In [88]:
# create a dictionary with the elements of all_features as keys that contains the number of occurrences of each element in the list corr_per_category_top
feature_prevalence = {feature: 0 for feature in features}
for i in range(len(categories)):
    top_corr = corr_per_category_top[i]
    if categories[i] in top_corr.index:
        top_corr = top_corr.drop(categories[i])
    for feature in top_corr.index:
        feature_prevalence[feature] += 1
print(feature_prevalence)

{'B_port_is_11095': 1, 'B_port_is_8080': 3, 'domain_is_else': 1, 'ds_field_B': 3, 'http_count_host': 1, 'http_count_req_content_type': 1, 'http_count_resp_content_type': 1, 'http_count_user_agents': 1, 'http_has_req_content_type': 1, 'http_has_resp_content_type': 1, 'http_has_user_agent': 1, 'is_http': 1, 'packet_inter_arrivel_A_min': 3, 'packet_inter_arrivel_B_min': 1, 'packet_inter_arrivel_firstQ': 3, 'packet_inter_arrivel_median': 3, 'packet_inter_arrivel_min': 3, 'packet_size_A_avg': 1, 'packet_size_A_stdev': 1, 'packet_size_A_thirdQ': 1, 'packet_size_A_var': 1, 'packet_size_B_avg': 1, 'packet_size_B_entropy': 1, 'packet_size_B_thirdQ': 1, 'packets_A_B_ratio': 1, 'reset': 1, 'reset_A': 1, 'reset_B': 2, 'ssl_count_client_ciphersuites': 1, 'ssl_dom_server_name_alexaRank': 1, 'ssl_ratio_client_elliptic_curves': 1, 'ssl_req_bytes_stdev': 1, 'ttl_A_avg': 2, 'ttl_A_firstQ': 4, 'ttl_A_max': 3, 'ttl_A_median': 4, 'ttl_A_min': 4, 'ttl_A_thirdQ': 4, 'ttl_B_avg': 1, 'ttl_B_firstQ': 1, 'ttl_B_

In [89]:
feature_corr_sum = {
    feature: sum(
        corr_per_category[category][feature]
        for category in range(len(categories))
        if feature in corr_per_category[category]
    )
    for feature in features
}
print(feature_corr_sum)

{'B_port_is_11095': 1.389501493255388, 'B_port_is_8080': 2.919231274741209, 'domain_is_else': 1.6345162598439504, 'ds_field_B': 2.895271529101306, 'http_count_host': 1.6287570854605373, 'http_count_req_content_type': 1.6287570854605373, 'http_count_resp_content_type': 1.6038172371216128, 'http_count_user_agents': 1.6287570854605373, 'http_has_req_content_type': 1.6287570854605373, 'http_has_resp_content_type': 1.6287570854605373, 'http_has_user_agent': 1.6287570854605373, 'is_http': 1.6287570854605373, 'packet_inter_arrivel_A_min': 2.7930045648305786, 'packet_inter_arrivel_B_min': 1.074807629398935, 'packet_inter_arrivel_firstQ': 2.906054256049794, 'packet_inter_arrivel_median': 2.8061393593210444, 'packet_inter_arrivel_min': 2.9027666830075507, 'packet_size_A_avg': 1.0808871311697275, 'packet_size_A_stdev': 1.3977990836263052, 'packet_size_A_thirdQ': 1.1161194721968493, 'packet_size_A_var': 1.278753706365447, 'packet_size_B_avg': 1.5647548563930362, 'packet_size_B_entropy': 1.81273976

In [96]:
feature_groups = {
    'B_port': [feature for feature in features if feature.startswith('B_port')],
    'http': [feature for feature in features if feature.startswith('http') or feature == 'is_http'],
    'packet_inter': [feature for feature in features if feature.startswith('packet_inter')],
    'packet_size': [feature for feature in features if feature.startswith('packet_size')],
    'reset': [feature for feature in features if feature.startswith('reset')],
    'ssl': [feature for feature in features if feature.startswith('ssl')],
    'ttl_A': [feature for feature in features if feature.startswith('ttl_A')],
    'ttl_B': [feature for feature in features if feature.startswith('ttl_B')],
    'ttl_others': [feature for feature in features if feature.startswith('ttl_') and feature not in ['ttl_A', 'ttl_B']],
    'others': []
}
# put all elements of features that are not in the above groups into 'others'
for feature in features:
    if feature not in feature_groups['B_port'] and feature not in feature_groups['http'] and feature not in feature_groups['packet_inter'] and feature not in feature_groups['packet_size'] and feature not in feature_groups['reset'] and feature not in feature_groups['ssl'] and feature not in feature_groups['ttl_A'] and feature not in feature_groups['ttl_B'] and feature not in feature_groups['ttl_others']:
        feature_groups['others'].append(feature)
print(feature_groups)
print(feature_groups['others'])


{'B_port': ['B_port_is_11095', 'B_port_is_8080'], 'http': ['http_count_host', 'http_count_req_content_type', 'http_count_resp_content_type', 'http_count_user_agents', 'http_has_req_content_type', 'http_has_resp_content_type', 'http_has_user_agent', 'is_http'], 'packet_inter': ['packet_inter_arrivel_A_min', 'packet_inter_arrivel_B_min', 'packet_inter_arrivel_firstQ', 'packet_inter_arrivel_median', 'packet_inter_arrivel_min'], 'packet_size': ['packet_size_A_avg', 'packet_size_A_stdev', 'packet_size_A_thirdQ', 'packet_size_A_var', 'packet_size_B_avg', 'packet_size_B_entropy', 'packet_size_B_thirdQ'], 'reset': ['reset', 'reset_A', 'reset_B'], 'ssl': ['ssl_count_client_ciphersuites', 'ssl_dom_server_name_alexaRank', 'ssl_ratio_client_elliptic_curves', 'ssl_req_bytes_stdev'], 'ttl_A': ['ttl_A_avg', 'ttl_A_firstQ', 'ttl_A_max', 'ttl_A_median', 'ttl_A_min', 'ttl_A_thirdQ'], 'ttl_B': ['ttl_B_avg', 'ttl_B_firstQ', 'ttl_B_max', 'ttl_B_median', 'ttl_B_min', 'ttl_B_stdev', 'ttl_B_thirdQ'], 'ttl_oth

In [97]:
feature_group_prevalence = {
    group: sum(feature_prevalence[feature] for feature in features)
    for group, features in feature_groups.items()
}
print(feature_group_prevalence)

{'B_port': 4, 'http': 8, 'packet_inter': 13, 'packet_size': 7, 'reset': 4, 'ssl': 4, 'ttl_A': 21, 'ttl_B': 8, 'ttl_others': 45, 'others': 5}


In [98]:
feature_group_corr_sum = {
    group: sum(
        feature_corr_sum[feature]
        for feature in features
    )
    for group, features in feature_groups.items()
}
print(feature_group_corr_sum)

{'B_port': 4.308732767996597, 'http': 13.005116835345373, 'packet_inter': 12.482772492607902, 'packet_size': 9.822016520311106, 'reset': 5.458403578218915, 'ssl': 5.484590364315682, 'ttl_A': 18.90515529339294, 'ttl_B': 12.233305454259588, 'ttl_others': 51.72858671784893, 'others': 7.188644964630882}


In [99]:
feature_group_prevalence_density = {
    group: feature_group_prevalence[group] / len(features)
    for group, features in feature_groups.items()
}
print(feature_group_prevalence_density)

{'B_port': 2.0, 'http': 1.0, 'packet_inter': 2.6, 'packet_size': 1.0, 'reset': 1.3333333333333333, 'ssl': 1.0, 'ttl_A': 3.5, 'ttl_B': 1.1428571428571428, 'ttl_others': 2.142857142857143, 'others': 1.6666666666666667}


In [100]:
feature_group_corr_sum_density = {
    group: feature_group_corr_sum[group] / len(features)
    for group, features in feature_groups.items()
}
print(feature_group_corr_sum_density)

{'B_port': 2.1543663839982985, 'http': 1.6256396044181716, 'packet_inter': 2.49655449852158, 'packet_size': 1.403145217187301, 'reset': 1.819467859406305, 'ssl': 1.3711475910789206, 'ttl_A': 3.15085921556549, 'ttl_B': 1.7476150648942268, 'ttl_others': 2.4632660341832824, 'others': 2.396214988210294}


<h3>Interactive widgets to explore feature importance</h3>

In [ ]:
# make a dataframe with the data in feature_prevalence and append a column with the data in feature_corr_sum
feature_analysis = pd.DataFrame.from_dict(feature_prevalence, orient='index', columns=['prevalence'])
feature_analysis.index.name = 'feature'
feature_analysis.reset_index(inplace=True)
feature_analysis['corr_sum'] = feature_analysis['feature'].map(feature_corr_sum)
feature_analysis.head()

,feature,prevalence,corr_sum
0,B_port_is_11095,1,1.389501
1,B_port_is_8080,3,2.919231
2,domain_is_else,1,1.634516
3,ds_field_B,3,2.895272
4,http_count_host,1,1.628757


In [103]:
feature_analysis['corr_avg'] = feature_analysis['corr_sum'] / feature_analysis['prevalence']

In [104]:
feature_analysis['group'] = feature_analysis['feature'].map(
    lambda x: next((group for group, features in feature_groups.items() if x in features), 'others')
)

In [118]:
feature_analysis

,feature,prevalence,corr_sum,corr_avg,group
0,B_port_is_11095,1,1.389501,1.389501,B_port
1,B_port_is_8080,3,2.919231,0.973077,B_port
2,domain_is_else,1,1.634516,1.634516,others
3,ds_field_B,3,2.895272,0.965091,others
4,http_count_host,1,1.628757,1.628757,http
5,http_count_req_content_type,1,1.628757,1.628757,http
6,http_count_resp_content_type,1,1.603817,1.603817,http
7,http_count_user_agents,1,1.628757,1.628757,http
8,http_has_req_content_type,1,1.628757,1.628757,http
9,http_has_resp_content_type,1,1.628757,1.628757,http


In [106]:
import plotly.express as px

In [124]:
fig_corr = px.sunburst(
    feature_analysis, 
    path=['group', 'feature'], 
    values='corr_sum',
    color='corr_avg', color_continuous_scale='Blues',
    title='Feature importance'
    )
fig_corr.show()